# 🦀 Day 3: Migrations and Connection Pooling

---

## What Are Database Migrations?

**Migrations** are versioned schema changes. Instead of manually altering tables in production, you:
- Write *up* scripts (apply changes)
- Write *down* scripts (revert changes)
- Track which migrations have been applied

This gives you reproducible, auditable schema evolution.

In [ ]:
:dep rusqlite = { version = "0.31", features = ["bundled"] }

use rusqlite::Connection;

## Migration Table

We need a table to track which migrations have been applied.

In [ ]:
use rusqlite::Connection;

fn ensure_migrations_table(conn: &Connection) -> rusqlite::Result<()> {
    conn.execute(
        "CREATE TABLE IF NOT EXISTS _migrations (version INTEGER PRIMARY KEY, name TEXT, applied_at TEXT)",
        [],
    )?;
    Ok(())
}

let conn = Connection::open_in_memory().unwrap();
ensure_migrations_table(&conn).unwrap();
println!("Migrations table ready.");

## Simple Migration Runner

A basic migration runner: each migration has a version, name, and up/down SQL.

In [ ]:
use rusqlite::Connection;
use std::time::{SystemTime, UNIX_EPOCH};

struct Migration {
    version: i32,
    name: String,
    up_sql: String,
    down_sql: String,
}

fn get_applied_versions(conn: &Connection) -> rusqlite::Result<Vec<i32>> {
    let mut stmt = conn.prepare("SELECT version FROM _migrations ORDER BY version")?;
    let versions = stmt.query_map([], |row| row.get(0))?.collect::<Result<Vec<_>, _>>()?;
    Ok(versions)
}

fn run_migration(conn: &Connection, m: &Migration) -> rusqlite::Result<()> {
    let tx = conn.transaction()?;
    tx.execute(&m.up_sql, [])?;
    let ts = SystemTime::now().duration_since(UNIX_EPOCH).unwrap().as_secs();
    tx.execute(
        "INSERT INTO _migrations (version, name, applied_at) VALUES (?1, ?2, ?3)",
        rusqlite::params![m.version, m.name, ts.to_string()],
    )?;
    tx.commit()?;
    Ok(())
}

let conn = Connection::open_in_memory().unwrap();
ensure_migrations_table(&conn).unwrap();

let migration = Migration {
    version: 1,
    name: "create_users".into(),
    up_sql: "CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT)".into(),
    down_sql: "DROP TABLE users".into(),
};

run_migration(&conn, &migration).unwrap();
println!("Migration 1 applied.");

let applied = get_applied_versions(&conn).unwrap();
println!("Applied versions: {:?}", applied);

## Running Pending Migrations

Apply only migrations that haven't been run yet.

In [ ]:
use rusqlite::Connection;

fn run_pending_migrations(conn: &Connection, migrations: &[Migration]) -> rusqlite::Result<()> {
    ensure_migrations_table(conn)?;
    let applied = get_applied_versions(conn)?;
    for m in migrations {
        if !applied.contains(&m.version) {
            run_migration(conn, m)?;
            println!("Applied: {} (v{})", m.name, m.version);
        }
    }
    Ok(())
}

let conn = Connection::open_in_memory().unwrap();
let migrations = vec![
    Migration {
        version: 1,
        name: "create_users".into(),
        up_sql: "CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT)".into(),
        down_sql: "DROP TABLE users".into(),
    },
    Migration {
        version: 2,
        name: "add_email".into(),
        up_sql: "ALTER TABLE users ADD COLUMN email TEXT".into(),
        down_sql: "".into(), // SQLite doesn't support DROP COLUMN easily
    },
];

run_pending_migrations(&conn, &migrations).unwrap();

// Verify schema: insert and select with new email column
conn.execute("INSERT INTO users (name, email) VALUES (?1, ?2)", rusqlite::params!["Alice", "alice@ex.com"]).unwrap();
let (name, email): (String, Option<String>) = conn.query_row("SELECT name, email FROM users WHERE id = 1", [], |r| Ok((r.get(0)?, r.get(1)?))).unwrap();
println!("User: {} | email: {:?}", name, email);

## Connection Pooling — Why?

Opening a DB connection is expensive. A **connection pool** maintains a set of reusable connections:
- **Why:** Avoid per-request connection overhead
- **What:** A collection of ready-to-use connections
- **How:** Check out a connection, use it, return it to the pool

## Simple Connection Pool (Vec + Arc<Mutex>)

A minimal pool: we store connections in a `Vec` and guard access with a `Mutex`.

In [ ]:
use rusqlite::Connection;
use std::sync::{Arc, Mutex};

struct SimplePool {
    connections: Arc<Mutex<Vec<Connection>>>,
}

impl SimplePool {
    fn new(size: usize, path: &str) -> rusqlite::Result<Self> {
        let mut connections = Vec::new();
        for _ in 0..size {
            connections.push(Connection::open(path)?);
        }
        Ok(SimplePool {
            connections: Arc::new(Mutex::new(connections)),
        })
    }

    fn get(&self) -> Option<Connection> {
        let mut guard = self.connections.lock().unwrap();
        guard.pop()
    }

    fn put(&self, conn: Connection) {
        let mut guard = self.connections.lock().unwrap();
        guard.push(conn);
    }
}

// Use :memory: — each Connection gets its own DB, so this is conceptual
let pool = SimplePool::new(2, ":memory:").unwrap();
let conn1 = pool.get().unwrap();
conn1.execute("CREATE TABLE test (id INTEGER)", []).unwrap();
pool.put(conn1);
let conn2 = pool.get().unwrap();
// conn2 is a different DB (different :memory: instance)
println!("Pool get/put works. (Note: :memory: per connection = separate DBs)");

## r2d2 / deadpool Patterns (Conceptual)

Production pools use crates like **r2d2** (sync) or **deadpool** (async):

```rust
// r2d2 pattern (conceptual)
let pool = r2d2::Pool::new(manager)?;
let conn = pool.get()?;  // Blocks until a connection is available
// use conn...
// conn is returned to pool when dropped (via PooledConnection)
```

```rust
// deadpool (async) pattern
let conn = pool.get().await?;
```

Key ideas: **blocking wait** when pool is exhausted, **automatic return** on drop, **configurable max size**.

## 🏋️ Exercise

Add a new migration (version 3) that adds a `created_at` column (TEXT, default current timestamp) to the `users` table. Run it and verify the column exists.

In [ ]:
:dep rusqlite = { version = "0.31", features = ["bundled"] }

use rusqlite::Connection;

// Reuse Migration struct and run_pending_migrations from above.
// Add migration version 3: ALTER TABLE users ADD COLUMN created_at TEXT DEFAULT (datetime('now'))

// YOUR CODE HERE

## 🎯 Key Takeaways

1. **Migrations** — versioned schema changes with up/down scripts
2. **Migration table** — track applied versions to avoid re-running
3. **Migration runner** — apply only pending migrations in order
4. **Connection pooling** — reuse connections to reduce overhead
5. **Simple pool** — Vec + Mutex; production uses r2d2/deadpool
6. **r2d2** — sync pool; **deadpool** — async pool

---

**Tomorrow:** Authentication — password hashing and JWT 🔐